# 14 — Prompt Evaluation

    ## Scenario and success criteria

    Northstar must decide whether a new support router is safe to release, including a critical boundary case.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Retain metric numerators and denominators.
- Compare a baseline and candidate on identical cases.
- Fail a release on any critical regression.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 14 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Aggregate accuracy can hide a critical failure in a small slice.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab14 import EvalCase, evaluate, intent_candidate, keyword_baseline, release_allowed

cases = [
    EvalCase("refund", "I want my money back", "refund", "direct"),
    EvalCase("login", "I cannot login", "technical", "direct"),
    EvalCase("boundary", "I read the refund policy; help fix a snapped handle", "technical", "boundary", "critical"),
    EvalCase("hours", "When do you open?", "other", "direct"),
]

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
baseline = evaluate(keyword_baseline, cases)
candidate = evaluate(intent_candidate, cases)
print("baseline", baseline)
print("candidate", candidate)
print("candidate release allowed:", release_allowed(candidate))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert baseline.total == candidate.total == 4
assert baseline.critical_failures == ("boundary",)
assert candidate.by_slice["boundary"] == (1, 1)
assert release_allowed(candidate)

## Production upgrade

Keep labelled cases versioned, review slice deltas, and use deterministic checks for schemas and forbidden outcomes. Semantic judges complement—not replace—these gates.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.